In [1]:
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

groq_api_key = "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"
tavily_api_key = os.getenv("TAVILY_API_KEY")

print("Groq loaded:", bool(groq_api_key))
print("Tavily loaded:", bool(tavily_api_key))

Groq loaded: True
Tavily loaded: True


In [2]:
#STATE BACKEND

import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from langchain_groq import ChatGroq

model = ChatGroq(
    model='qwen/qwen3.6-27b',
    api_key= groq_api_key,
)

agent1 = create_deep_agent(
    model=model,
    backend = StateBackend(),
)

agent2 = create_deep_agent(
    model=model,
)

result = agent2.invoke({
    "messages":[{
        "role":"user",
        "content":(
            "create a file at /notes/todo.txt with exactly this content:\n"
            "my name is sarthak \n"
            "then do let me know when u do the same"
        )
    }]
})


In [3]:
print(result['messages'][-1].content)

I've created the file at `/notes/todo.txt` with the exact content you requested. Let me know if you need anything else!


In [4]:
result

{'messages': [HumanMessage(content='create a file at /notes/todo.txt with exactly this content:\nmy name is sarthak \nthen do let me know when u do the same', additional_kwargs={}, response_metadata={}, id='71270338-837c-4f0e-837d-6d56a28a7d0a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants me to create a file at `/notes/todo.txt` with the content "my name is sarthak". I need to:\n1. First check if the `/notes` directory exists\n2. Create the file with the specified content\n3. Confirm when done\n\nLet me create the directory if needed and then write the file.\n', 'tool_calls': [{'id': 'pv9mfzhy5', 'function': {'arguments': '{"content":"my name is sarthak","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 2757, 'total_tokens': 2878, 'completion_time': 0.267290183, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.204032243, 

files are stored in that langgraph state

In [7]:
follow  = agent1.invoke({
    "messages":result["messages"] + [{
        "role":"user",
        "content":"read /notes/todo/.txt back to me"
    }],
    "files":result.get('files',{}),
})

In [8]:
print(follow['messages'][-1].content)

Here is the content of `/notes/todo.txt`:

`my name is sarthak`


### file system backend local disk

In [9]:
from deepagents.backends import FilesystemBackend

ROOT = "."

agent3 = create_deep_agent(
    model = model,
    backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True)
)

follow1 = agent3.invoke({
    "messages":[{
        "role":"user",
        "content":(
            "create a file at /notes/todo.txt with exactly this content:\n"
            "my name is sarthak \n"
            "then do let me know when u do the same"
        )
    }]
})

print(follow1['messages'][-1].content)

I've created the file at `/notes/todo.txt` with the exact content you requested. Let me know if you need anything else!


In [11]:
agent4 = create_deep_agent(
    model = model,
    backend = FilesystemBackend(root_dir=ROOT,virtual_mode=True),
)

follow2 = agent4.invoke({
    "messages":[{
        "role":"user",
        "content":"read /notes/todo.txt back to me verbatim"
    }]
})

print(follow2['messages'][-1].content)

my name is sarthak


StoreBackend verification

creates a deep agent backed by a langraph store invokes it to write a file on one thread then proves the backend works by reading that file back on a different thread - something state backend cant do

In [12]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

In [13]:
agent5 = create_deep_agent(
    model=model,
    backend=StoreBackend(
        namespace = lambda rt:("demo-user",),
    ),
    store=store,
)